# Практическая работа 15 (Тест)

## Настройка окружения

In [1]:
import pandas as pd
import numpy as np

Наша задача обучить модель машинного обучения, которая будет предсказывать дефолт. Предсказание дефолта - одна из первых задач в банковской сфере, решением которой занимались с использованием алгоритмов машинного обучения. Нужно по данным, которые предоставил потенциальный заемщик, определить, будет у него дефолт или нет. На вход модель будет принимать данные о клиенте, а на выходе она должна работать в двух режимах:

- выдавать вероятность дефолта для данного клиента,
- выдавать правильный с точки зрения модели класс клиента (есть у него дефолт или нет).
- выдавать правильный с точки зрения модели класс клиента (есть у него дефолт или нет).

Выборка была разбита на две части для обучения и для тестирования модели. В обучающей выборке 50000 клиентов, в тестовой выборке - 49000. Выполните скачивание двух выборок для дальнейшего анализа.

## Вопрос 1

Загрузите обучающий набор данных и выведите данные описательной статистики. Выберите верные утверждения.

1. В нашей выборке только у 6% клиентов есть дефолт
2. Среднемесячный доход = 5400$
3. У столбца NumberOfDependents больше половины значений - нулевые.
4. Столбцы MonthlyIncome и NumberOfDependents содержат пропущенные значения

In [2]:
# Загрузка данных
training = pd.read_csv('training_data_defolt.csv')
training.head()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,0,0.250476,65,0,1963.000000,NaN,5,0,2,0,0.0
1,0,0.975402,58,0,0.049441,5905.0,1,0,0,0,0.0
2,0,0.000000,82,0,0.000933,3215.0,10,0,0,0,0.0
3,0,0.571491,36,0,0.113864,2300.0,6,1,0,0,0.0
4,0,0.410499,45,0,0.478572,7816.0,11,0,1,0,1.0


In [3]:
# Вывод описательной статистики
print("Описательная статистика:")
print(training.describe())
print("\n" + "="*50)
print("\nИнформация о данных:")
print(training.info())
print("\n" + "="*50)
print("\nПервые строки данных:")
print(training.head())

Описательная статистика:
       SeriousDlqin2yrs  RevolvingUtilizationOfUnsecuredLines           age  \
count      50000.000000                          50000.000000  50000.000000   
mean           0.066860                              7.927880     52.240520   
std            0.249782                            332.393142     14.766593   
min            0.000000                              0.000000     21.000000   
25%            0.000000                              0.030096     41.000000   
50%            0.000000                              0.154426     52.000000   
75%            0.000000                              0.555651     63.000000   
max            1.000000                          50708.000000    109.000000   

       NumberOfTime30-59DaysPastDueNotWorse      DebtRatio  MonthlyIncome  \
count                          50000.000000   50000.000000   4.014700e+04   
mean                               0.428220     352.441921   6.642232e+03   
std                             

In [4]:
# Проверка утверждения 1: В нашей выборке только у 6% клиентов есть дефолт
training['SeriousDlqin2yrs'].mean() == 0.06

np.False_

In [5]:
# Проверка утверждения 2: Среднемесячный доход = 5400$
training['MonthlyIncome'].mean() == 5400

np.False_

In [6]:
# Проверка утверждения 3: У столбца NumberOfDependents больше половины значений - нулевые
(training['NumberOfDependents'] == 0).mean() > 0.5

np.True_

In [7]:
# Проверка утверждения 4: Столбцы MonthlyIncome и NumberOfDependents содержат пропущенные значения
training['MonthlyIncome'].isna().any() and training['NumberOfDependents'].isna().any()

np.True_

Рассчитаем средние значения признаков в обучающей выборке, и заполним полученными числами пропуски как в тестовом наборе данных, так и в самой обучающей выборке. Мы будем заполнять средними значениями из обучающей выборки, так как при решении реальной задачи нам будут доступны только данные для обучения. Далее сохраним в переменную X значения всех предикторов, в переменную y - целевую переменную `SeriousDlqin2yrs`.

## Вопрос 2

Выполним стандартизацию факторов с помощью библиотеки `sklearn` и сохраним результат в переменную `X_zscore`. Обратите внимание, что объект `scaler` будет использован в дальнейшем для преобразования факторов тестовой выборки.

```py
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_zscore = ...
```

В качестве ответа укажите шестое значение в векторе объекта с индексом 7, округлённое до 2 знаков после запятой.

In [8]:
# Рассчитываем средние значения признаков в обучающей выборке (исключая целевую переменную)
target_col = 'SeriousDlqin2yrs'
feature_cols = [col for col in training.columns if col != target_col]

# Средние значения только для числовых признаков
mean_values = training[feature_cols].select_dtypes(include=[np.number]).mean()

print("Средние значения признаков:")
print(mean_values)

Средние значения признаков:
RevolvingUtilizationOfUnsecuredLines       7.927880
age                                       52.240520
NumberOfTime30-59DaysPastDueNotWorse       0.428220
DebtRatio                                352.441921
MonthlyIncome                           6642.232222
NumberOfOpenCreditLinesAndLoans            8.476040
NumberOfTimes90DaysLate                    0.269920
NumberRealEstateLoansOrLines               1.017500
NumberOfTime60-89DaysPastDueNotWorse       0.246040
NumberOfDependents                         0.756180
dtype: float64


In [9]:
# Заполняем пропуски в обучающей выборке средними значениями (создаем новые столбцы)
training_filled = training.copy()

for col in feature_cols:
    new_col_name = f'{col}_filled'
    if col in mean_values.index:
        # Для числовых признаков заполняем пропуски средними значениями
        training_filled[new_col_name] = training[col].fillna(mean_values[col])
    else:
        # Для нечисловых признаков просто копируем (или можно оставить как есть)
        training_filled[new_col_name] = training[col]

print(f"Создано {len([col for col in training_filled.columns if col.endswith('_filled')])} новых столбцов")
print(f"\nКоличество пропусков в исходных данных:")
missing = training[feature_cols].isna().sum()
print(missing[missing > 0] if len(missing[missing > 0]) > 0 else "Нет пропусков")

Создано 10 новых столбцов

Количество пропусков в исходных данных:
MonthlyIncome         9853
NumberOfDependents    1333
dtype: int64


In [10]:
# Загружаем тестовый набор данных
test = pd.read_csv('test_data_defolt.csv')

# Заполняем пропуски в тестовом наборе теми же средними значениями из обучающей выборки
test_filled = test.copy()

for col in feature_cols:
    if col in test.columns:
        new_col_name = f'{col}_filled'
        if col in mean_values.index:
            # Для числовых признаков заполняем пропуски средними значениями из обучающей выборки
            test_filled[new_col_name] = test[col].fillna(mean_values[col])
        else:
            # Для нечисловых признаков просто копируем
            test_filled[new_col_name] = test[col]

print(f"Создано {len([col for col in test_filled.columns if col.endswith('_filled')])} новых столбцов в тестовых данных")
print(f"\nКоличество пропусков в тестовых данных (исходные):")
missing_test = test[feature_cols].isna().sum() if all(col in test.columns for col in feature_cols) else pd.Series()
print(missing_test[missing_test > 0] if len(missing_test[missing_test > 0]) > 0 else "Нет пропусков")

Создано 10 новых столбцов в тестовых данных

Количество пропусков в тестовых данных (исходные):
MonthlyIncome         7456
NumberOfDependents     979
dtype: int64


In [11]:
# Создаем X (все предикторы) и y (целевая переменная) из обучающей выборки
# Используем заполненные столбцы
X_cols = [col for col in training_filled.columns if col.endswith('_filled')]
X = training_filled[X_cols].copy()
y = training[target_col].copy()

print(f"Размерность X: {X.shape}")
print(f"Размерность y: {y.shape}")
print(f"\nСтолбцы в X:")
print(X.columns.tolist())

Размерность X: (50000, 10)
Размерность y: (50000,)

Столбцы в X:
['RevolvingUtilizationOfUnsecuredLines_filled', 'age_filled', 'NumberOfTime30-59DaysPastDueNotWorse_filled', 'DebtRatio_filled', 'MonthlyIncome_filled', 'NumberOfOpenCreditLinesAndLoans_filled', 'NumberOfTimes90DaysLate_filled', 'NumberRealEstateLoansOrLines_filled', 'NumberOfTime60-89DaysPastDueNotWorse_filled', 'NumberOfDependents_filled']


In [12]:
# Стандартизация факторов с помощью StandardScaler
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_zscore = scaler.fit_transform(X)

print(f"Размерность X_zscore: {X_zscore.shape}")
print(f"Первая строка X_zscore: {X_zscore[0]}")
print(f"Второе значение в векторе объекта с индексом 0: {X_zscore[0][1]:.2f}")

# Ответ: второе значение в векторе объекта с индексом 5, округлённое до 2 знаков после запятой
print(f"\nВектор объекта с индексом 7: {X_zscore[7]}")
answer = round(X_zscore[7][5], 2)
print(f"Ответ: {answer}")

Размерность X_zscore: (50000, 10)
Первая строка X_zscore: [-2.30975919e-02  8.64086085e-01 -1.00303244e-01  8.02707985e-01
  5.60068215e-17 -6.71996706e-01 -6.36143520e-02  8.77122457e-01
 -5.81515694e-02 -6.87670163e-01]
Второе значение в векторе объекта с индексом 0: 0.86

Вектор объекта с индексом 7: [-0.02385115 -0.01628828  0.13392973 -0.17543259  0.05793262  0.48793823
 -0.06361435  1.76986796 -0.05815157 -0.68767016]
Ответ: 0.49


## Вопрос 3

Выполните создание и обучение модели логистической регрессии:

```py
from sklearn.linear_model import LogisticRegression
model_lr = LogisticRegression()
```

Оценим качество обучения модели на данных тестовой выборки.\
Предварительно:

1. Заполните пропуски средними значениями, рассчитанными ранее на данных обучающей выборки.
2. Сохраните отдельно в переменную `X_test` набор значений предикторов, `y_test` - значения целевой переменной.
3. Выполните стандартизацию значений X_test на основе объекта `scaler`, созданного ранее. Так как объект `scaler` уже обучен (знает среднее и стандартное отклонение каждого признака), то для преобразования необходимо использовать метод `transform`.

В качестве ответа укажите значение метрики `accuracy`, округлённое до 2 знаков после запятой.

In [13]:
# Создание и обучение модели логистической регрессии
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression()
model_lr.fit(X_zscore, y)

print("Модель обучена")

Модель обучена


In [14]:
# Подготовка тестовых данных
# 1. Пропуски уже заполнены средними значениями из обучающей выборки в test_filled
# 2. Создаем X_test и y_test
X_test_cols = [col for col in test_filled.columns if col.endswith('_filled')]
X_test = test_filled[X_test_cols].copy()
y_test = test[target_col].copy()

print(f"Размерность X_test: {X_test.shape}")
print(f"Размерность y_test: {y_test.shape}")

Размерность X_test: (37500, 10)
Размерность y_test: (37500,)


In [15]:
# 3. Стандартизация X_test с помощью обученного scaler
X_test_zscore = scaler.transform(X_test)

print(f"Размерность X_test_zscore: {X_test_zscore.shape}")

Размерность X_test_zscore: (37500, 10)


In [16]:
# Оценка качества модели на тестовой выборке
from sklearn.metrics import accuracy_score

y_pred = model_lr.predict(X_test_zscore)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy на тестовой выборке: {accuracy:.4f}")

Accuracy на тестовой выборке: 0.9329


In [17]:
# Ответ: значение метрики accuracy, округлённое до 2 знаков после запятой
answer = round(accuracy, 2)
print(answer)

0.93


## Вопрос 4

Из значения точности мы никак не можем понять, сколько меток каждого класса правильно предсказала модель. В нашей задаче мало значений с классом 1 (дефолт), но много 0 (возврат кредита). Может быть такая ситуация, когда модель очень хорошо научилась выделять характеристики большого класса, в нашем случае 0, но совсем не умеет выделять характеристики маленького класса. А часто именно последние в большей степени интересуют аналитиков.

Другой способ оценивать качество работы классификатора - использовать таблицу сопряженности (`confusion matrix`). Постройте `confusion matrix` с использованием библиотеки `sklearn`. В качестве ответа укажите кол-во истинно положительных (TP) объектов.

In [18]:
# Построение confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)
print(f"\nСтруктура confusion matrix:")
print("TN (True Negatives) - [0,0]:", cm[0, 0])
print("FP (False Positives) - [0,1]:", cm[0, 1])
print("FN (False Negatives) - [1,0]:", cm[1, 0])
print("TP (True Positives) - [1,1]:", cm[1, 1])

Confusion Matrix:
[[34880    93]
 [ 2424   103]]

Структура confusion matrix:
TN (True Negatives) - [0,0]: 34880
FP (False Positives) - [0,1]: 93
FN (False Negatives) - [1,0]: 2424
TP (True Positives) - [1,1]: 103


In [19]:
# Ответ: количество истинно отрицательных (TN) объектов
TN = cm[0, 0]
print(TN)

34880


## Вопрос 5

Укажите средневзвешенное значение метрики `precision` по двум классам (weighted avg). Значение округлите до 2 знаков после запятой.

In [20]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      1.00      0.97     34973
           1       0.53      0.04      0.08      2527

    accuracy                           0.93     37500
   macro avg       0.73      0.52      0.52     37500
weighted avg       0.91      0.93      0.91     37500



## Вопрос 6

Укажите вероятность дефолта, округлённую до 2 знаков после запятой, для данного клиента:

```py
borrower = pd.Series({
    'RevolvingUtilizationOfUnsecuredLines': 0.4,
    'age': 25,
    'DebtRatio': 2,
    'MonthlyIncome': 5000,
    'NumberOfOpenCreditLinesAndLoans': 5,
    'NumberRealEstateLoansOrLines': 0,
    'NumberOfTime30-59DaysPastDueNotWorse': 0,
    'NumberOfTime60-89DaysPastDueNotWorse': 0,
    'NumberOfTimes90DaysLate': 2,
    'NumberOfDependents': 3
})
```

Не забудьте выполнить стандартизацию признаков с помощью обученного объекта `scaler`.

In [21]:
# Предсказание вероятности дефолта для нового клиента
borrower = pd.Series({
    'RevolvingUtilizationOfUnsecuredLines': 0.005,
    'age': 28,
    'DebtRatio': 15,
    'MonthlyIncome': 1300,
    'NumberOfOpenCreditLinesAndLoans': 3,
    'NumberRealEstateLoansOrLines': 0,
    'NumberOfTime30-59DaysPastDueNotWorse': 1,
    'NumberOfTime60-89DaysPastDueNotWorse': 3,
    'NumberOfTimes90DaysLate': 2,
    'NumberOfDependents': 0
})

# Преобразуем в DataFrame с правильными столбцами (с суффиксом _filled)
# Важно: используем тот же порядок столбцов, что и в X
borrower_filled = pd.DataFrame(index=[0])

for col in X.columns:
    original_col = col.replace('_filled', '')
    if original_col in borrower.index:
        borrower_filled[col] = borrower[original_col]
    else:
        # Если столбца нет, используем среднее значение из обучающей выборки
        borrower_filled[col] = mean_values[original_col] if original_col in mean_values.index else 0

# Убеждаемся, что порядок столбцов точно соответствует X
borrower_filled = borrower_filled[X.columns]

print("Данные клиента (заполненные):")
print(borrower_filled)
print(f"\nПорядок столбцов соответствует X: {list(borrower_filled.columns) == list(X.columns)}")

Данные клиента (заполненные):
   RevolvingUtilizationOfUnsecuredLines_filled  age_filled  \
0                                        0.005        28.0   

   NumberOfTime30-59DaysPastDueNotWorse_filled  DebtRatio_filled  \
0                                          1.0              15.0   

   MonthlyIncome_filled  NumberOfOpenCreditLinesAndLoans_filled  \
0                1300.0                                     3.0   

   NumberOfTimes90DaysLate_filled  NumberRealEstateLoansOrLines_filled  \
0                             2.0                                  0.0   

   NumberOfTime60-89DaysPastDueNotWorse_filled  NumberOfDependents_filled  
0                                          3.0                        0.0  

Порядок столбцов соответствует X: True


In [22]:
# Стандартизация признаков с помощью обученного scaler
# scaler был обучен на X, поэтому порядок столбцов должен точно совпадать
borrower_zscore = scaler.transform(borrower_filled)

print("Стандартизированные данные клиента:")
print(borrower_zscore)
print(f"Размерность: {borrower_zscore.shape}")

Стандартизированные данные клиента:
[[-0.02383611 -1.6415948   0.13392973 -0.16818228 -0.32897547 -1.05864168
   0.40774273 -0.90836855  0.65089862 -0.68767016]]
Размерность: (1, 10)


In [23]:
# Получение вероятностей дефолта
# model_lr был обучен на X_zscore, поэтому используем стандартизированные данные
probabilities = model_lr.predict_proba(borrower_zscore)
prob_default = probabilities[0][1]  # Вероятность класса 1 (дефолт)

print(f"Вероятности для классов: {probabilities[0]}")
print(f"Вероятность класса 0 (нет дефолта): {probabilities[0][0]:.4f}")
print(f"Вероятность класса 1 (дефолт): {prob_default:.4f}")

Вероятности для классов: [0.97238338 0.02761662]
Вероятность класса 0 (нет дефолта): 0.9724
Вероятность класса 1 (дефолт): 0.0276


In [24]:
# Ответ: вероятность дефолта, округлённая до 2 знаков после запятой
answer = round(prob_default, 2)
print(answer)

0.03


## Вопрос 7

Выполните построение модели случайного леса:

```py
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier(random_state=0, n_jobs=-1)
model_rf.fit(X_zscore, y)
```

Рассчитайте и запомните значение метрики accuracy, округлённое до 3 знаков после запятой (оно пригодится в следующем задании). Для расчёта воспользуйтесь функцией accuracy_score из пакета sklearn.metric.

Оцените значимость признаков. В качестве ответа укажите значимость для признака age, округлённую до 2 знаков после запятой.

In [25]:
# Построение модели случайного леса
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(random_state=0, n_jobs=-1)
model_rf.fit(X_zscore, y)

print("Модель случайного леса обучена")

Модель случайного леса обучена


In [26]:
# Расчёт accuracy на тестовой выборке
y_pred_rf = model_rf.predict(X_test_zscore)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
accuracy_rf_rounded = round(accuracy_rf, 3)

print(f"Accuracy модели случайного леса: {accuracy_rf:.6f}")
print(f"Accuracy (округлённое до 3 знаков): {accuracy_rf_rounded}")

Accuracy модели случайного леса: 0.934533
Accuracy (округлённое до 3 знаков): 0.935


In [27]:
# Оценка значимости признаков
feature_importances = model_rf.feature_importances_

# Создаём DataFrame для удобного просмотра
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': feature_importances
}).sort_values('importance', ascending=False)

print("Значимость признаков:")
print(importance_df)

# Находим значимость для признака age
monthly_income_col = 'age_filled'
monthly_income_importance = feature_importances[list(X.columns).index(monthly_income_col)]

print(f"\nЗначимость для признака {monthly_income_col}: {monthly_income_importance:.6f}")

Значимость признаков:
                                       feature  importance
0  RevolvingUtilizationOfUnsecuredLines_filled    0.187966
3                             DebtRatio_filled    0.169020
4                         MonthlyIncome_filled    0.142878
1                                   age_filled    0.127689
5       NumberOfOpenCreditLinesAndLoans_filled    0.094783
6               NumberOfTimes90DaysLate_filled    0.094134
2  NumberOfTime30-59DaysPastDueNotWorse_filled    0.054727
9                    NumberOfDependents_filled    0.046936
8  NumberOfTime60-89DaysPastDueNotWorse_filled    0.045330
7          NumberRealEstateLoansOrLines_filled    0.036537

Значимость для признака age_filled: 0.127689


In [28]:
# Ответ: значимость для признака age, округлённая до 2 знаков после запятой
answer = round(monthly_income_importance, 2)
print(answer)

0.13


## Вопрос 8

Выполните отбop признаков с показателем значимости выше или равным пороговому значению 0.05 и заново
осуществите обучение модели случайного леса с использованием отобранных признаков. В качестве ответа
укажите, на сколько повысилось значение метрики `accuracy`. Ответ округлите до 3 знаков после запятой.

**_Примечание_**: для расчёта значения метрики `accuracy` воспользуйтесь функцией `accuracy_score` из
пакета `sklearn.metric`.

In [29]:
# Отбор признаков с показателем значимости >= 0.05
threshold = 0.05
selected_features = importance_df[importance_df['importance'] >= threshold]['feature'].tolist()

print(f"Отобрано признаков с значимостью >= {threshold}: {len(selected_features)}")
print("Отобранные признаки:")
print(selected_features)

Отобрано признаков с значимостью >= 0.05: 7
Отобранные признаки:
['RevolvingUtilizationOfUnsecuredLines_filled', 'DebtRatio_filled', 'MonthlyIncome_filled', 'age_filled', 'NumberOfOpenCreditLinesAndLoans_filled', 'NumberOfTimes90DaysLate_filled', 'NumberOfTime30-59DaysPastDueNotWorse_filled']


In [30]:
# Создание новых наборов данных с отобранными признаками
# Нужно найти индексы отобранных признаков в исходных данных
selected_indices = [list(X.columns).index(feat) for feat in selected_features]

# Для обучающей выборки используем X_zscore (уже стандартизированные данные)
X_zscore_selected = X_zscore[:, selected_indices]

# Для тестовой выборки используем X_test_zscore (уже стандартизированные данные)
X_test_zscore_selected = X_test_zscore[:, selected_indices]

print(f"Размерность X_zscore_selected: {X_zscore_selected.shape}")
print(f"Размерность X_test_zscore_selected: {X_test_zscore_selected.shape}")

Размерность X_zscore_selected: (50000, 7)
Размерность X_test_zscore_selected: (37500, 7)


In [31]:
# Обучение модели случайного леса с использованием отобранных признаков
model_rf_selected = RandomForestClassifier(random_state=0, n_jobs=-1)
model_rf_selected.fit(X_zscore_selected, y)

print("Модель случайного леса с отобранными признаками обучена")

Модель случайного леса с отобранными признаками обучена


In [32]:
# Расчёт accuracy на тестовой выборке для модели с отобранными признаками
y_pred_rf_selected = model_rf_selected.predict(X_test_zscore_selected)
accuracy_rf_selected = accuracy_score(y_test, y_pred_rf_selected)

print(f"Accuracy модели с отобранными признаками: {accuracy_rf_selected:.6f}")
print(f"Accuracy исходной модели: {accuracy_rf:.6f}")
print(f"Разница (повышение): {accuracy_rf_selected - accuracy_rf:.6f}")

Accuracy модели с отобранными признаками: 0.933627
Accuracy исходной модели: 0.934533
Разница (повышение): -0.000907


In [33]:
# Ответ: на сколько повысилось значение метрики accuracy, округлённое до 3 знаков после запятой
accuracy_improvement = round(accuracy_rf_selected - accuracy_rf, 3)
print(accuracy_improvement)

-0.001


## Вопрос 9

Укажите средневзвешенное значение метрики `recall` по двум классам (`weighted avg`). Значение округлите до 2 знаков после запятой.

In [34]:
print(classification_report(y_test, y_pred_rf_selected))

              precision    recall  f1-score   support

           0       0.94      0.99      0.97     34973
           1       0.52      0.17      0.26      2527

    accuracy                           0.93     37500
   macro avg       0.73      0.58      0.61     37500
weighted avg       0.91      0.93      0.92     37500



## Вопрос 10

Выполните построение модели k-ближайших соседей. Для поиска оптимальных значений гиперпараметров используйте метод `Randomized Search` с параметрами `n_iter=5, cv=3, random_state=0` и данной сеткой гиперпараметров:

```py
param_grid = {
    'n_neighbors': list(range(1,31)),
    'metric': ['cosine', 'euclidean', 'manhattan'],
    'weights': ['uniform', 'distance']
}
```

В качестве ответа укажите оптимальное значение параметра `weights`.

In [35]:
# Построение модели k-ближайших соседей с поиском оптимальных гиперпараметров
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV

# Создаём модель KNN
knn = KNeighborsClassifier()

# Сетка гиперпараметров
param_grid = {
    'n_neighbors': list(range(1,31)),
    'metric': ['cosine', 'euclidean', 'manhattan'],
    'weights': ['uniform', 'distance']
}

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    knn, 
    param_grid, 
    n_iter=5, 
    cv=3, 
    random_state=0,
    n_jobs=-1
)

# Обучение на отобранных признаках
random_search.fit(X_zscore_selected, y)

print("Поиск оптимальных гиперпараметров завершён")
print(f"Лучшие параметры: {random_search.best_params_}")
print(f"Лучший score: {random_search.best_score_:.4f}")

Поиск оптимальных гиперпараметров завершён
Лучшие параметры: {'weights': 'uniform', 'n_neighbors': 24, 'metric': 'euclidean'}
Лучший score: 0.9350


In [36]:
# Ответ: оптимальное значение параметра weights
random_search.best_params_['weights']

'uniform'

## Вопрос 11

Укажите средневзвешенное значение метрики `recall` по двум классам (`weighted avg`). Значение округлите до 2 знаков после запятой.

In [37]:
# Получаем предсказания на тестовой выборке
y_pred_knn = random_search.predict(X_test_zscore_selected)

print(classification_report(y_test, y_pred_knn))

              precision    recall  f1-score   support

           0       0.94      1.00      0.97     34973
           1       0.59      0.09      0.15      2527

    accuracy                           0.93     37500
   macro avg       0.76      0.54      0.56     37500
weighted avg       0.91      0.93      0.91     37500



## Вопрос 12

Выберите выводы, которые можно сделать на основании проведённого анализа.

1. [✓] В наших данных наблюдался дисбаланс классов.
2. [✗] Самого высокого среднеклассового значения (macro avg) метрики f1-score удалось достичь модели логистической регрессии.
3. [✗] В нашей задаче высокое значение метрики accuracy позволяет нам с уверенностью говорить о качественной работе модели.
4. [✓] Все три полученные модели показали высокое качество работы по метрике accuracy.

In [38]:
# Анализ результатов для проверки утверждений

print("=== Анализ утверждений ===\n")

# Утверждение 1: Дисбаланс классов
print("1. Дисбаланс классов:")
print(f"   Класс 0 (нет дефолта): {34973} объектов ({34973/37500*100:.1f}%)")
print(f"   Класс 1 (дефолт): {2527} объектов ({2527/37500*100:.1f}%)")
print(f"   ВЫВОД: Дисбаланс классов ОЧЕВИДЕН ✓\n")

# Утверждение 2: Macro avg f1-score
print("2. Macro avg f1-score по моделям:")
print(f"   Логистическая регрессия: macro avg f1-score = 0.52")
print(f"   Случайный лес (отобранные признаки): macro avg f1-score = 0.61")
print(f"   KNN: macro avg f1-score = 0.56")
print(f"   ВЫВОД: Самый высокий у случайного леса (0.61), не у логистической регрессии ✗\n")

# Утверждение 3: Высокая accuracy = качественная модель?
print("3. Высокая accuracy и качество модели:")
print(f"   Логистическая регрессия: accuracy = {accuracy:.4f}, recall класса 1 = 0.04")
print(f"   Случайный лес: accuracy = {accuracy_rf:.4f}, recall класса 1 = 0.17")
print(f"   KNN: accuracy ≈ 0.93, recall класса 1 = 0.09")
print(f"   ВЫВОД: При дисбалансе высокая accuracy обманчива - модель плохо находит дефолты ✗\n")

# Утверждение 4: Все модели показали высокую accuracy
print("4. Accuracy всех моделей:")
print(f"   Логистическая регрессия: {accuracy:.4f}")
print(f"   Случайный лес: {accuracy_rf:.4f}")
print(f"   KNN: ≈ 0.93")
print(f"   ВЫВОД: Все три модели показали высокую accuracy (≈0.93) ✓")

=== Анализ утверждений ===

1. Дисбаланс классов:
   Класс 0 (нет дефолта): 34973 объектов (93.3%)
   Класс 1 (дефолт): 2527 объектов (6.7%)
   ВЫВОД: Дисбаланс классов ОЧЕВИДЕН ✓

2. Macro avg f1-score по моделям:
   Логистическая регрессия: macro avg f1-score = 0.52
   Случайный лес (отобранные признаки): macro avg f1-score = 0.61
   KNN: macro avg f1-score = 0.56
   ВЫВОД: Самый высокий у случайного леса (0.61), не у логистической регрессии ✗

3. Высокая accuracy и качество модели:
   Логистическая регрессия: accuracy = 0.9329, recall класса 1 = 0.04
   Случайный лес: accuracy = 0.9345, recall класса 1 = 0.17
   KNN: accuracy ≈ 0.93, recall класса 1 = 0.09
   ВЫВОД: При дисбалансе высокая accuracy обманчива - модель плохо находит дефолты ✗

4. Accuracy всех моделей:
   Логистическая регрессия: 0.9329
   Случайный лес: 0.9345
   KNN: ≈ 0.93
   ВЫВОД: Все три модели показали высокую accuracy (≈0.93) ✓
